In [1]:
print()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC, LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import re
from scipy.sparse import hstack, csr_matrix

In [ ]:
import gdown
import pickle
import os

In [ ]:
def load_from_drive(drive_url, pkl=False, save_local=True, local_filename=None):

    file_id = drive_url.split('/d/')[1].split('/')[0]
    download_url = f"https://drive.google.com/uc?id={file_id}"

    if not local_filename:
        local_filename = "downloaded_file.pkl" if pkl else "downloaded_file.csv"

    print(f" Downloading from Google Drive...\nURL: {drive_url}")
    gdown.download(download_url, local_filename, quiet=False)


    if pkl:
        print(f"Loading Pickle/Joblib file → {local_filename}")
        try:
            return pickle.load(open(local_filename, "rb"))  # try normal pickle first
        except Exception:
            return joblib.load(local_filename)  # fallback to joblib
    else:
        print(f"📄 Loading CSV file → {local_filename}")
        return pd.read_csv(local_filename)


# **LOAD THE DICT DATASET**

In [ ]:
url_csv = "https://drive.google.com/file/d/1s1ZcPiWtJouk2aKQqfBoYfVxygKNtdZi/view?usp=drive_link"
df = load_from_drive(url_csv, pkl=False, local_filename="dataset.csv")
print(" Loaded CSV:", df.shape)


URL: https://drive.google.com/file/d/1s1ZcPiWtJouk2aKQqfBoYfVxygKNtdZi/view?usp=drive_link


Downloading...
From: https://drive.google.com/uc?id=1s1ZcPiWtJouk2aKQqfBoYfVxygKNtdZi
To: /content/dataset.csv
100%|██████████| 13.5M/13.5M [00:00<00:00, 41.8MB/s]


📄 Loading CSV file → dataset.csv
 Loaded CSV: (163133, 3)


In [ ]:
df

,tamil_word,eng_word,pos
0,நான்,I,pronoun
1,சுவையான,delicious,adjective
2,உணவு,food,noun
3,சாப்பிடுகிறேன்,eat,verb
4,அவள்,she,pronoun
...,...,...,...
163128,ஹ்2 காலன் முழ்ல் 120 காலன் வரை அளவுள்ள பாறை,puncheon,none
163129,ஹ்4 துப்பாக்கிகளை உடைய போர்க்கப்பல் வகை,seventy-four,noun
163130,ஹ்க்ஷ் சீட்டுள்ள கட்டுவைத்து ஆடும் ஆட்டம்,"taroc, tarot",none
163131,ஹ்க்ஷ்ஹ்-ஆம் ஆண்டுகளில் நடைபெற்ற அனைத்துக் கிற...,nicence,adjective


In [ ]:
df.columns

Index(['tamil_word', 'eng_word', 'pos'], dtype='object')

In [ ]:
# Step 2: Build Tamil-English dictionary from dataset

tamil_to_english_dict = {}
for idx, row in df.iterrows():
    tamil_word = str(row['tamil_word']).strip()
    english_word = str(row['eng_word']).strip()
    pos_tag = str(row["pos"]).strip()


    if tamil_word not in tamil_to_english_dict:
        tamil_to_english_dict[tamil_word] = []
    tamil_to_english_dict[tamil_word].append({
        'english': english_word,
        'pos': pos_tag
    })

print(f"\nBuilt dictionary with {len(tamil_to_english_dict)} Tamil words")

# Create reverse mapping for analysis
#english_to_pos = {}
#for idx, row in df.iterrows():
    #english_word = str(row['eng_word']).strip()
    #pos_tag = str(row["pos"]).strip()
    #if english_word not in english_to_pos:
        #english_to_pos[english_word] = pos_tag


Built dictionary with 163121 Tamil words


# **LOAD THE CREATED DATASET**

In [ ]:
url_csv1 = "https://drive.google.com/file/d/1C_hdcPUlqo1bNhAyJJQTQdvY2A8F_4te/view?usp=drive_link"
df_smt = load_from_drive(url_csv1, pkl=False, local_filename="dataset1.csv")
print(" Loaded CSV:", df_smt.shape)


URL: https://drive.google.com/file/d/1C_hdcPUlqo1bNhAyJJQTQdvY2A8F_4te/view?usp=drive_link


Downloading...
From: https://drive.google.com/uc?id=1C_hdcPUlqo1bNhAyJJQTQdvY2A8F_4te
To: /content/dataset1.csv
100%|██████████| 16.8M/16.8M [00:00<00:00, 48.9MB/s]


📄 Loading CSV file → dataset1.csv
 Loaded CSV: (362042, 3)


In [ ]:
df_smt

,Tamil_Word,English_Word,Frequency
0,ராஜாவாகிய,king,264
1,ராஜாவாகிய,of,26
2,ராஜாவாகிய,nebuchadnezzar,9
3,ஆகாஸ்,ahaz,15
4,ஆகாஸ்,urijah,3
...,...,...,...
362037,போக்குகொண்டவர்கள்,there,1
362038,போக்குகொண்டவர்கள்,was,1
362039,போக்குகொண்டவர்கள்,also,1
362040,குறைவுள்ளவர்கள்,commissioners,1


In [ ]:
# Build dictionary: Tamil → most frequent English
smt_tamil_eng_dict = (
    df_smt.sort_values("Frequency", ascending=False)
          .drop_duplicates(subset=["Tamil_Word"])
          .set_index("Tamil_Word")["English_Word"]
          .to_dict()
)

print(f" SMT dictionary built with {len(smt_tamil_eng_dict)} unique Tamil words")


 SMT dictionary built with 208076 unique Tamil words


# **LOAD THE POS TAG MODEL**

In [ ]:
import joblib

url_pkl_model = "https://drive.google.com/file/d/1FaLP_ONMdeo7MYoouvTUf3qKtsjW_ZKe/view?usp=drive_link"
url_pkl_le = "https://drive.google.com/file/d/1A6Pi9MQiUo_zhDz7ImaEb3vDZYddkDUB/view?usp=drive_link"

# Load pipeline and label encoder
loaded_pipeline = load_from_drive(url_pkl_model, pkl=True, local_filename="model.pkl")
le = load_from_drive(url_pkl_le, pkl=True, local_filename="label_encoder.pkl")

URL: https://drive.google.com/file/d/1FaLP_ONMdeo7MYoouvTUf3qKtsjW_ZKe/view?usp=drive_link


Downloading...
From: https://drive.google.com/uc?id=1FaLP_ONMdeo7MYoouvTUf3qKtsjW_ZKe
To: /content/model.pkl
100%|██████████| 2.45M/2.45M [00:00<00:00, 22.5MB/s]
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/bas

Loading Pickle/Joblib file → model.pkl


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


URL: https://drive.google.com/file/d/1A6Pi9MQiUo_zhDz7ImaEb3vDZYddkDUB/view?usp=drive_link


Downloading...
From: https://drive.google.com/uc?id=1A6Pi9MQiUo_zhDz7ImaEb3vDZYddkDUB
To: /content/label_encoder.pkl
100%|██████████| 580/580 [00:00<00:00, 1.32MB/s]

Loading Pickle/Joblib file → label_encoder.pkl



/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
# Step 2.5: Add custom Tamil-English mappings

def add_custom_mappings(mappings_dict):

    global tamil_to_english_dict

    for tamil_word, english_word in mappings_dict.items():
        tamil_word = tamil_word.strip()
        english_word = str(english_word).strip()

        # Predict POS tag for the English word
        predicted_pos = get_pos_tag(english_word)

        if tamil_word not in tamil_to_english_dict:
            tamil_to_english_dict[tamil_word] = []

        # Add mapping with predicted POS
        tamil_to_english_dict[tamil_word].append({
            'english': english_word,
            'pos': predicted_pos
        })

        print(f"  {tamil_word:20} -> {english_word:20} (predicted: {predicted_pos})")

    print(f"\n✓ Added {len(mappings_dict)} custom mappings")
    print(f"✓ Total words in dictionary: {len(tamil_to_english_dict)}\n")

def remove_mapping(tamil_word):
    """Remove a specific Tamil word mapping"""
    tamil_word = tamil_word.strip()
    if tamil_word in tamil_to_english_dict:
        del tamil_to_english_dict[tamil_word]
        print(f"✓ Removed '{tamil_word}' from dictionary")
    else:
        print(f"✗ '{tamil_word}' not found in dictionary")

def view_dictionary():
    """View all current mappings in the dictionary"""
    if not tamil_to_english_dict:
        print("Dictionary is empty!")
        return

    print("\nCurrent Tamil-English Dictionary:")
    print("="*80)
    for tamil_word, translations in sorted(tamil_to_english_dict.items()):
        for trans in translations:
            print(f"  {tamil_word:20} -> {trans['english']:20} ({trans['pos']})")
    print("="*80 + "\n")

In [ ]:
from scipy.sparse import hstack, csr_matrix
import numpy as np


def determine_article(word, pos_tag):
    """Determine if article is needed"""
    if pos_tag in ["PROPN", "pronoun", "determiner", "verb", "adverb", "adjective", "conjunction", "SCONJ", "PUNCT", "auxiliary"]:
        return None

    uncountable = ["food", "water", "air", "information", "knowledge", "time", "money", "rice", "milk"]
    if word.lower() in uncountable:
        return None

    if pos_tag == "NOUN":
        if word[0].lower() in "aeiou":
            return "an"
        else:
            return "a"

    return None

def conjugate_verb(subject, verb):
    """Conjugate verb based on subject pronoun"""
    subject_lower = subject.lower()

    if subject_lower in ["i", "you", "we", "they"]:
        return verb
    elif subject_lower in ["he", "she", "it"]:# third person singular
        if verb.endswith("y"):
            return verb[:-1] + "is"
        elif verb.endswith(("s", "ss", "x", "z", "ch", "sh")):
            return verb + "es"
        elif verb.endswith("o"):
            return verb + "es"
        else:
            return verb + "s"

    return verb

def build_sentence(components, words, pos_tags):
    """Build fluent English sentence from components"""
    sentence_parts = []

    # Subject
    if components["subject"]:
        subject = components["subject"][0][0]
        sentence_parts.append(subject)

    # Auxiliary verbs (modals, helping verbs)
    if components["auxiliary"]:
        sentence_parts.append(components["auxiliary"][0][0])

    # Main verb (conjugated)
    if components["verb"]:
        verb = components["verb"][0][0]
        if components["subject"]:
            subject = components["subject"][0][0]
            verb = conjugate_verb(subject, verb)
        sentence_parts.append(verb)

    # Adjectives + Objects (with articles)
    adj_words = [adj[0] for adj in components["adjective"]]
    for obj_word, obj_tamil in components["object"]:
        # Check if adjective comes before this noun
        if adj_words:
            sentence_parts.extend(adj_words)
            adj_words = []

        # Add article if needed
        article = determine_article(obj_word, "NOUN")
        if article:
            sentence_parts.append(article)
        sentence_parts.append(obj_word)

    # Prepositions and their objects
    if components["preposition"]:
        for prep_word, prep_tamil in components["preposition"]:
            sentence_parts.append(prep_word)

    # Adverbs
    if components["adverb"]:
        for adv_word, adv_tamil in components["adverb"]:
            sentence_parts.append(adv_word)

    # Conjunctions
    if components["conjunction"]:
        sentence_parts.append(components["conjunction"][0][0])

    # Particles
    if components["particle"]:
        for part_word, part_tamil in components["particle"]:
            sentence_parts.append(part_word)

    return sentence_parts

def reorganize_sentence(words, pos_tags, tamil_words):

    components = {
        "subject": [],
        "verb": [],
        "object": [],
        "adjective": [],
        "adverb": [],
        "preposition": [],
        "determiner": [],
        "conjunction": [],
        "auxiliary": [],
        "particle": [],
        "unknown": []
    }

    for word, pos, tamil_word in zip(words, pos_tags, tamil_words):
        if pos.lower() == "pronoun":
            components["subject"].append((word, tamil_word))
        elif pos.lower() == "verb":
            components["verb"].append((word, tamil_word))
        elif pos.lower() == "noun":
            components["object"].append((word, tamil_word))
        elif pos.lower() == "adjective":
            components["adjective"].append((word, tamil_word))
        elif pos.lower() == "adverb":
            components["adverb"].append((word, tamil_word))
        elif pos.lower() == "preposition":
            components["preposition"].append((word, tamil_word))
        elif pos.lower() == "determiner":
            components["determiner"].append((word, tamil_word))
        elif pos.lower() in ["conjunction", "SCONJ"]:
            components["conjunction"].append((word, tamil_word))
        elif pos.lower() == "auxiliary":
            components["auxiliary"].append((word, tamil_word))
        elif pos.lower() == "PART":
            components["particle"].append((word, tamil_word))
        else:
            components["unknown"].append((word, tamil_word))

    return components


def add_features(df):
    df['word_len'] = df['eng_word'].apply(len)
    df['prefix_2'] = df['eng_word'].str[:2]
    df['prefix_3'] = df['eng_word'].str[:3]
    df['suffix_2'] = df['eng_word'].str[-2:]
    df['suffix_3'] = df['eng_word'].str[-3:]
    return df

def get_pos_tag(word):
    try:
        # Convert word to lowercase for consistency
        word_lower = word.lower()
        pronouns = {
            "i", "me", "you", "he", "him", "she", "her",
            "it", "we", "us", "they", "them",
            "my", "your", "his", "her", "its", "our", "their",
            "mine", "yours", "hers", "ours", "theirs",
            "myself", "yourself", "himself", "herself", "itself",
            "ourselves", "yourselves", "themselves",
            "who", "whom", "whose", "which", "that"
        }

        if word_lower in pronouns:
            return "PRONOUN"

        # Create a temporary DataFrame with feature columns
        temp_df = pd.DataFrame({'eng_word': [word_lower]})

        # Use your add_features() function to generate morphological features
        temp_df = add_features(temp_df)

        # Predict POS using your full trained pipeline (includes TF-IDF, OneHot, Scaler, etc.)
        prediction_encoded = loaded_pipeline.predict(temp_df)[0]

        # Convert encoded label to string using the LabelEncoder
        pos_tag = le.inverse_transform([prediction_encoded])[0]

        return pos_tag.upper()

    except Exception as e:
        print(f" Error predicting '{word}': {e}")
        return "NOUN"  # default fallback




def translate_word(tamil_word):
    tamil_word = tamil_word.strip()

    # Check SMT-trained dictionary
    if tamil_word in smt_tamil_eng_dict:
        eng_word = smt_tamil_eng_dict[tamil_word]
        return eng_word,get_pos_tag(eng_word), True

    #  Check refined manual dictionary
    elif tamil_word in tamil_to_english_dict:
        entry = tamil_to_english_dict[tamil_word][0]
        return entry['english'], entry['pos'], True

    else:
        return tamil_word, "unknown", False

# **Main translation function**

In [ ]:
# Step 5: Main translation function

def translate_tamil_to_english(tamil_sentence, debug=False):


    tamil_words = tamil_sentence.split()

    if debug:
        print("\n" + "="*70)
        print("TRANSLATION DEBUG")
        print("="*70)
        print(f"Tamil input: {tamil_sentence}")
        print(f"Tamil words: {tamil_words}\n")

    # Translate each word
    english_words = []
    pos_tags = []
    found_flags = []

    for tamil_word in tamil_words:
        english_word, pos_tag, found = translate_word(tamil_word)
        english_words.append(english_word)
        pos_tags.append(pos_tag)
        found_flags.append(found)

        #debug
        status = "✓ Found" if found else "✗ NOT in dataset"
        if debug:
            print(f"{tamil_word:20} -> {english_word:20} ({pos_tag:10}) {status}")

    #debug
    if debug:
        print("\n" + "-"*70)
        print(f"Coverage: {sum(found_flags)}/{len(found_flags)} words found in dataset")

    # Check if translation is possible
    if sum(found_flags) == 0:
        if debug:
            print("WARNING: No words found in dataset!")
        return "[Translation not possible - words not in dataset]"

    # Reorganize components
    components = reorganize_sentence(english_words, pos_tags, tamil_words)

    if debug:
        print("\nSentence Components:")
        for comp_type, items in components.items():
            if items:
                words_list = [w[0] for w in items]
                print(f"  {comp_type:15}: {words_list}")

    # Build sentence
    sentence_parts = build_sentence(components, english_words, pos_tags)

    if sentence_parts:
        sentence = " ".join(sentence_parts)
        # Clean up multiple spaces
        sentence = " ".join(sentence.split())
        # Capitalize first letter
        sentence = sentence[0].upper() + sentence[1:] if len(sentence) > 1 else sentence.upper()
        # Add period
        if not sentence.endswith(('.', '!', '?', ']')):
            sentence += '.'
    else:
        sentence = "[Translation failed - unable to structure sentence]"

    if debug:
        print(f"\nGenerated sentence: {sentence}")
        print("="*70 + "\n")

    return sentence

In [ ]:
# Step 6: Batch translation

def translate_batch(tamil_sentences, debug=False):
    """Translate multiple Tamil sentences"""
    translations = []
    for tamil_sent in tamil_sentences:
        english_sent = translate_tamil_to_english(tamil_sent, debug=debug)
        translations.append({
            "Tamil": tamil_sent,
            "English": english_sent
        })
    return pd.DataFrame(translations)

In [ ]:
# Step 7: Coverage analysis

def analyze_coverage(tamil_sentences):
    """Analyze how many words from sentences are in the dataset"""
    print("="*70)
    print("DATASET COVERAGE ANALYSIS")
    print("="*70)
    results = []
    for tamil_sent in tamil_sentences:
        tamil_words = tamil_sent.split()
        found = sum(1 for w in tamil_words if w in tamil_to_english_dict)
        coverage = (found / len(tamil_words) * 100) if tamil_words else 0
        results.append({
            "Tamil": tamil_sent,
            "Total words": len(tamil_words),
            "Found": found,
            "Coverage %": f"{coverage:.1f}%"
        })
    return pd.DataFrame(results)

# **TESTING**

In [ ]:

if __name__ == "__main__":

    # Step 1: Add custom Tamil-English mappings
    #print("\n" + "="*70)
    #print("ADDING CUSTOM MAPPINGS")
    #print("="*70)

    custom_mappings = {
        'நான்': 'I',
        'சாப்பிடுகிறேன்': 'eat',
        'சுவையான': 'delicious',
        'உணவு': 'food',
        'அவள்': 'she',
        'பெரிய': 'big',
        'சிவப்பு': 'red',
        'கார்': 'car',
        'ஓட்டுகிறாள்': 'drive',
        'பள்ளிக்குச்': 'school',
        'செல்கிறேன்': 'go',
        'தம்பி': 'brother',
        'விளையாடுகிறான்': 'play',
    }

    #add_custom_mappings(custom_mappings)

    # Step 2: View dictionary
    #view_dictionary()

    # Step 3: Coverage analysis
    test_sentences = [
      "நான் சுவையான உணவு சாப்பிடுகிறேன்",
      "அவள் பெரிய சிவப்பு கார் ஓட்டுகிறாள்",
      "நான் பள்ளிக்குச் செல்கிறேன்",
      "சர்வதேச நாணய நிதியம் இலங்கைக்கு கடன் வழங்கினால் இதே போன்ற நிபந்தனைகள் திணிக்கப்படும் ",
      "நீங்கள் உங்கள் அடுத்த பதவி உயர்வு பெற வேண்டும்",
      "அவர் கால்பந்து விளையாடிய பின்னர் வீட்டிற்கு வந்தார்"

]

    #coverage_df = analyze_coverage(test_sentences)
    #print(coverage_df.to_string(index=False))

    # Step 4: Single translation with debug
    #print("\n\nExample Translation (with debug):")
    #result = translate_tamil_to_english(test_sentences[0])

    # Step 5: Batch translation
    print("\n" + "="*70)
    print("BATCH TRANSLATION RESULTS")
    print("="*70)
    results_df = translate_batch(test_sentences)
    print(results_df.to_string(index=False))


BATCH TRANSLATION RESULTS
                                                                                 Tamil                                           English
                                                      நான் சுவையான உணவு சாப்பிடுகிறேன்                             I eat delicious food.
                                                   அவள் பெரிய சிவப்பு கார் ஓட்டுகிறாள்                   She driveses red a major a car.
                                                           நான் பள்ளிக்குச் செல்கிறேன்                               I a school a gonna.
சர்வதேச நாணய நிதியம் இலங்கைக்கு கடன் வழங்கினால் இதே போன்ற நிபந்தனைகள் திணிக்கப்படும்  Imf international hike a sri a same a conditions.
                                        நீங்கள் உங்கள் அடுத்த பதவி உயர்வு பெற வேண்டும்                                You increase next.
                                   அவர் கால்பந்து விளையாடிய பின்னர் வீட்டிற்கு வந்தார்                     He home a football a . after.


In [ ]:
import sacrebleu

predicted = results_df['English'].tolist()

#  Reference translations (ground truth)
reference = [[
    "I eat delicious food.",
    "She drives a big red car.",
    "I am going to school.",
    "If the IMF gives a loan to Sri Lanka, similar conditions will be imposed.",
    "You should get your next promotion.",
    "He played football then came home."
]]

# Compute corpus-level metrics
bleu = sacrebleu.corpus_bleu(predicted, reference)
chrf = sacrebleu.corpus_chrf(predicted, reference)
ter = sacrebleu.corpus_ter(predicted, reference)

print(" Translation Quality Metrics:")
print("--------------------------------")
print(" BLEU Score:", round(bleu.score, 2))
print(" ChrF Score:", round(chrf.score, 2))
print(" TER (lower is better):", round(ter.score, 2))

#  Sentence-level BLEU for detailed inspection
print("\n Sentence-level BLEU breakdown:")
for i, (pred, ref) in enumerate(zip(predicted, reference[0])):
    sent_bleu = sacrebleu.sentence_bleu(pred, [ref])
    print(f"{i+1}. BLEU: {round(sent_bleu.score, 2)}")
    print(f"   Pred: {pred}")
    print(f"   Ref : {ref}\n")


 Translation Quality Metrics:
--------------------------------
 BLEU Score: 13.81
 ChrF Score: 35.61
 TER (lower is better): 73.17

 Sentence-level BLEU breakdown:
1. BLEU: 100.0
   Pred: I eat delicious food.
   Ref : I eat delicious food.

2. BLEU: 13.89
   Pred: She driveses red a major a car.
   Ref : She drives a big red car.

3. BLEU: 10.68
   Pred: I a school a gonna.
   Ref : I am going to school.

4. BLEU: 3.03
   Pred: Imf international hike a sri a same a conditions.
   Ref : If the IMF gives a loan to Sri Lanka, similar conditions will be imposed.

5. BLEU: 9.93
   Pred: You increase next.
   Ref : You should get your next promotion.

6. BLEU: 7.81
   Pred: He home a football a . after.
   Ref : He played football then came home.



In [ ]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.5 MB/s eta 0:00:00


# **GEC CORRECTION**

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# Path to model folder in Drive
model_path = r"/content/drive/MyDrive/grammar_correction_t5"


tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
    device_map="auto" if torch.cuda.is_available() else None
)

print(" Model loaded successfully from Google Drive!")


`torch_dtype` is deprecated! Use `dtype` instead!


 Model loaded successfully from Google Drive!


In [ ]:
results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Tamil    6 non-null      object
 1   English  6 non-null      object
dtypes: object(2)
memory usage: 228.0+ bytes


In [ ]:
def correct_sentence(text):
    try:
        input_text = "grammar: " + text
        input_ids = tokenizer.encode(input_text, return_tensors="pt").to(model.device)
        outputs = model.generate(
            input_ids,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )
        corrected = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return corrected
    except Exception as e:
        print(f"Error correcting: {text} ({e})")
        return text  # fallback — return same text if something fails


# ---------------------------
# Example DataFrame (replace with your real one)
# ---------------------------
# df = pd.DataFrame({
#     'Tamil': ['அவள் பள்ளிக்கு போனாள்', 'நான் சாப்பிட்டேன்', 'அவன் ஓடினான்', 'அவள் பாடுகிறாள்'],
#     'English': ['She go to school.', 'I eat food.', 'He run fast.', 'She sing song.']
# })

# ---------------------------
# Generate corrected sentences
# ---------------------------
results_df["Corrected_English"] = results_df["English"].apply(correct_sentence)

# ---------------------------
# Display Tamil + original + corrected English
# ---------------------------
for idx, row in results_df.iterrows():
    print(f"\n Tamil: {row['Tamil']}")
    print(f" Original English: {row['English']}")
    print(f" Corrected English: {row['Corrected_English']}")



 Tamil: நான் சுவையான உணவு சாப்பிடுகிறேன்
 Original English: I eat delicious food.
 Corrected English: I eat delicious food.

 Tamil: அவள் பெரிய சிவப்பு கார் ஓட்டுகிறாள்
 Original English: She driveses red a major a car.
 Corrected English: She drives red a major a car.

 Tamil: நான் பள்ளிக்குச் செல்கிறேன்
 Original English: I a school a gonna.
 Corrected English: I am a school a gonna.

 Tamil: சர்வதேச நாணய நிதியம் இலங்கைக்கு கடன் வழங்கினால் இதே போன்ற நிபந்தனைகள் திணிக்கப்படும் 
 Original English: Imf international hike a sri a same a conditions.
 Corrected English: Imf international hike a sri a sri a same conditions.

 Tamil: நீங்கள் உங்கள் அடுத்த பதவி உயர்வு பெற வேண்டும்
 Original English: You increase next.
 Corrected English: You increase next to your grammar.

 Tamil: அவர் கால்பந்து விளையாடிய பின்னர் வீட்டிற்கு வந்தார்
 Original English: He home a football a . after.
 Corrected English: He home a football a. after.
